# APIs de LLM — práctica (Sesión 1)

**OpenAI:** **2a** — Responses API (referencia principal). **2b** — Chat Completions (patrón multi‑proveedor / LiteLLM). **3** — Anthropic Messages API (guía completa: `system`, `messages`, `max_tokens`, respuesta, costes, errores).

Python 3.11+, `uv sync`, claves en Colab (secretos) o entorno. **No subas API keys a Git.**


## OpenAI: dos APIs, una recomendación (criterio del máster)

- **Responses API** (`client.responses.create`): la interfaz **más reciente y recomendada** para proyectos nuevos. Unifica capacidades, usa `instructions` + `input`, `output_text`, gestión de estado (`previous_response_id`) y herramientas nativas. **En este programa es la API principal;** los ejercicios 2a siguen esta estructura.
- **Chat Completions** (`client.chat.completions.create`): API **anterior**, soportada indefinidamente pero **ya no recomendada** para desarrollos nuevos solo en OpenAI. Su patrón `messages` con roles (`system`, `user`, `assistant`) es el que comparten **Anthropic, Google, Mistral**, etc., y el que aparecerá al trabajar **abstracción de proveedores** y agregadores como **LiteLLM**.

**Colab:** `%pip install -q openai`. Secreto `OPENAI_API_KEY` con acceso al notebook. **Local:** `export OPENAI_API_KEY=...`.

## Ejercicio 2a — Responses API: llamada completa y piezas

### 1. Llamada completa (diferencias frente a Chat Completions)

- **`instructions`**: rol, reglas y restricciones (equivalente al *system prompt*), **parámetro de primer nivel**, no es un mensaje más en el hilo.
- **`input`**: entrada del usuario — **string** en un solo turno o **lista de mensajes** con roles (`user`, `assistant`, `developer`) para multi-turno o más control.
- **Salida:** `response.output_text` (atajo del SDK); en casos avanzados (herramientas, varios bloques) se recorre `response.output`.

### 2. Contexto multi-turno

- **Manual:** incluir todo el historial en `input` como lista (igual que reconstruir mensajes en Chat Completions).
- **`previous_response_id`:** encadena la respuesta anterior; suele usarse con **`store=True`** para que OpenAI guarde y recupere contexto (implicaciones de **privacidad** si hay datos sensibles — entonces `store=False` e historial manual).

### 3. Parámetros clave

`model`, `temperature`, **`max_output_tokens`** (tope de generación; si se alcanza, `status` puede ser `incomplete`), **`store`** (persistencia en servidores de OpenAI).

### 4. Respuesta: contenido, estado y metadatos

- **`status`:** `completed` (esperado), `incomplete` (revisar `incomplete_details`, p. ej. límite de salida), `failed` (`error`).
- **`usage`:** `input_tokens`, `output_tokens`, `total_tokens`; en modelos con razonamiento, detalle en `output_tokens_details`.

### 5. Coste orientativo

Los precios cambian: [API pricing](https://openai.com/api-pricing). El ejemplo en código usa tarifas de ejemplo para `gpt-4o-mini` (revisa la web antes de usar en producción).

### 6. Errores

El SDK expone excepciones tipadas (`AuthenticationError`, `RateLimitError`, `BadRequestError`, …). El código siguiente incluye un **try/except** mínimo y comprobación de `status`.

Documentación: [Responses](https://platform.openai.com/docs/guides/text).

In [ ]:
# %pip install -q openai

import os
from getpass import getpass

from openai import (
    APIConnectionError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)


def obtener_api_key_openai() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    return getpass("OPENAI_API_KEY: ").strip()


# Precios orientativos USD por 1M tokens (gpt-4o-mini) — actualiza desde la web de OpenAI
PRICING_GPT4O_MINI = {"input": 0.15, "output": 0.60}

api_key = obtener_api_key_openai()
if not api_key:
    raise ValueError("Falta OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

instructions = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

input_usuario = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

try:
    respuesta = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=input_usuario,
        temperature=0.3,
        max_output_tokens=500,
        store=False,
    )
except AuthenticationError:
    print("401: revisa OPENAI_API_KEY (válida y activa).")
    raise
except RateLimitError:
    print("429: límite de tasa o crédito; espera o revisa facturación.")
    raise
except BadRequestError as e:
    print("400: petición mal formada:", getattr(e, "message", e))
    raise
except (APIConnectionError, InternalServerError) as e:
    print("Red o error en servidor OpenAI:", e)
    raise

print("--- Contenido (output_text) ---")
print(respuesta.output_text)

print("--- Estado ---")
print("status:", respuesta.status)
if respuesta.status == "incomplete" and respuesta.incomplete_details:
    print("incomplete_details:", respuesta.incomplete_details)
if respuesta.status == "failed" and respuesta.error:
    print("error:", respuesta.error)

print("--- Metadatos ---")
print("id:", respuesta.id)
print("model (snapshot real):", respuesta.model)

if respuesta.usage:
    u = respuesta.usage
    print("usage:", u.model_dump())
    inp = u.input_tokens
    out = u.output_tokens
    p = PRICING_GPT4O_MINI
    coste = (inp / 1_000_000) * p["input"] + (out / 1_000_000) * p["output"]
    print(f"coste aprox. (USD): ${coste:.6f}")

# --- Opcional: mismo modelo con input como lista (multi-turno manual) ---
# respuesta_2 = client.responses.create(
#     model=MODEL,
#     instructions=instructions,
#     input=[
#         {"role": "user", "content": "¿Qué es una API REST?"},
#         {"role": "assistant", "content": "Una API REST es una interfaz basada en HTTP y recursos..."},
#         {"role": "user", "content": "¿En qué se diferencia de GraphQL?"},
#     ],
#     temperature=0.3,
#     max_output_tokens=400,
#     store=False,
# )
# print(respuesta_2.output_text)

# --- Opcional: encadenar con previous_response_id (requiere store=True en la 1ª llamada) ---
# Ver guía del máster; implica almacenamiento en OpenAI.

## Ejercicio 2b — Chat Completions: patrón `messages` (legacy OpenAI, estándar multi‑proveedor)

La **Chat Completions API** (`client.chat.completions.create`) es la API **anterior** de OpenAI: **sigue soportada**, pero **ya no es la recomendada** para desarrollos nuevos *solo* en OpenAI.

Su estructura de **`messages`** con roles **`system`**, **`user`**, **`assistant`** es el patrón que comparten **Anthropic, Google, Mistral** y muchos agregadores; por eso **sigue siendo imprescindible entenderla**. La verás al trabajar **abstracción de proveedores** y herramientas como **LiteLLM** (interfaz tipo Chat Completions).

### Desglose del `system` (equivalente conceptual a `instructions` en Responses)

1. **Rol:** quién es el modelo (“Eres un consultor senior…”).
2. **Instrucciones operativas:** qué hacer y cómo (idioma, tono, formato).
3. **Restricciones:** qué evitar (inventar datos, viñetas innecesarias, etc.).

### Diferencias clave respecto a Responses (resumen)

| Concepto | Responses API | Chat Completions |
|----------|---------------|------------------|
| Rol / reglas globales | `instructions` (top-level) | mensaje `role: "system"` |
| Entrada usuario (1 turno) | `input` (string o lista) | `messages` con `user` |
| Texto principal de salida | `output_text` | `choices[0].message.content` |
| Tope de salida | `max_output_tokens` | `max_tokens` |
| Estado explícito | `status`, `incomplete_details` | `finish_reason` en la choice |

Documentación: [Chat Completions](https://platform.openai.com/docs/guides/text-generation).

In [ ]:
# %pip install -q openai

import os
from getpass import getpass

from openai import (
    APIConnectionError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)


def obtener_api_key_openai() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    return getpass("OPENAI_API_KEY: ").strip()


PRICING_GPT4O_MINI = {"input": 0.15, "output": 0.60}

api_key = obtener_api_key_openai()
if not api_key:
    raise ValueError("Falta OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

system_content = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

user_content = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

try:
    respuesta = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ],
        temperature=0.3,
        max_tokens=500,
    )
except AuthenticationError:
    print("401: revisa OPENAI_API_KEY.")
    raise
except RateLimitError:
    print("429: límite de tasa o crédito.")
    raise
except BadRequestError as e:
    print("400:", getattr(e, "message", e))
    raise
except (APIConnectionError, InternalServerError) as e:
    print("Red o servidor:", e)
    raise

choice = respuesta.choices[0]
msg = choice.message.content

print("--- Contenido (choices[0].message.content) ---")
print(msg)

print("--- Metadatos ---")
print("finish_reason:", choice.finish_reason)
print("id:", respuesta.id)
print("model:", respuesta.model)

if respuesta.usage:
    u = respuesta.usage
    print("usage:", u.model_dump())
    inp = u.prompt_tokens
    out = u.completion_tokens
    p = PRICING_GPT4O_MINI
    coste = (inp / 1_000_000) * p["input"] + (out / 1_000_000) * p["output"]
    print(f"coste aprox. (USD): ${coste:.6f}")

# Inspección opcional del JSON completo (útil en depuración):
# print(respuesta.model_dump_json(indent=2)[:2000])

## Ejercicio 3 — Anthropic Messages API (`messages.create`)

### Contexto: una sola API

Anthropic expone **una única** interfaz para texto generado: la **Messages API**. A diferencia de OpenAI (Responses + Chat Completions en paralelo), aquí **toda** la funcionalidad consolidada pasa por `client.messages.create`.

El patrón recuerda a **Chat Completions**: array `messages` con roles y respuesta estructurada. Las diferencias clave frente a OpenAI: **`system` en parámetro aparte** (como `instructions` en Responses), **`max_tokens` obligatorio**, salida como **bloques** en `content` (no hay `output_text`), y detalles de parámetros (`temperature` 0–1, `top_k`, etc.).

**Colab:** `%pip install -q anthropic`, secreto `ANTHROPIC_API_KEY` con acceso al notebook. **Local:** `export ANTHROPIC_API_KEY=...`.

Documentación: [Messages](https://docs.anthropic.com/en/api/messages).

### 1. Llamada completa

- `model`, `system`, `messages`, **`max_tokens` (obligatorio)**, `temperature`.
- Texto típico: `response.content[0].text` o iterar bloques (herramientas, *extended thinking*, etc.).

### 2. `system`: rol, instrucciones y restricciones

Igual que `instructions` en OpenAI Responses: **primer nivel**, separado del hilo. Puede ser **string** o **lista de bloques** (`type: text`, `cache_control` para *prompt caching* en sesiones posteriores). En los ejercicios basta el string.

### 3. `messages`: conversación y roles

Solo **`user`** y **`assistant`** dentro del array: **no** existe `system` dentro de `messages`.

**Alternancia estricta:** el primer mensaje debe ser `user`; no pueden ir dos `user` seguidos. Para varias piezas de contexto en un turno, usa un solo `user` con **bloques** de `content` o texto concatenado.

La API es **stateless**: no hay equivalente a `previous_response_id`; el historial se **reenvía entero** en cada llamada (impacto en **tokens** y coste). Anthropic mitiga repetición con **prompt caching** (sesiones posteriores).

### 4. Parámetros esenciales y otros

| Parámetro | Notas |
|-----------|--------|
| `max_tokens` | **Obligatorio**. Si se agota, `stop_reason == "max_tokens"`. |
| `temperature` | 0.0–**1.0** (no hasta 2.0 como OpenAI). |
| `top_p` / `top_k` | No mezclar `temperature` y `top_p`; `top_k` es propio de Anthropic. |
| `stop_sequences` | Cortan la generación si el modelo las emite. |

*Extended thinking* (modelos compatibles): parámetro `thinking` con presupuesto de tokens; la respuesta puede incluir bloques `thinking` facturados como salida.

### 5. Respuesta: contenido, `stop_reason`, metadatos

- **`content`**: lista de bloques; lo habitual es un bloque `text`.
- **`stop_reason`:** `end_turn` (ok), `max_tokens` (truncado), `stop_sequence`, `tool_use`, etc.
- **`id`**, **`model`** (coincide con el id de modelo pedido), **`type`**, **`role`**. No viene **timestamp** en la respuesta: regístralo en tu código si lo necesitas.

### 6. `usage` y coste

`input_tokens` y `output_tokens`; **no** hay `total_tokens` en la respuesta — súmalos tú. Con *prompt caching* aparecen campos extra (`cache_creation_input_tokens`, `cache_read_input_tokens`, …).

Precios: [Anthropic pricing](https://www.anthropic.com/pricing). El código usa tarifas de **ejemplo** para `claude-haiku-4-5-20251001` (revísalas antes de producción).

### 7. Errores y reintentos del SDK

Excepciones tipadas (`AuthenticationError`, `RateLimitError`, `BadRequestError`, …). El cliente **reintenta** por defecto errores transitorios (p. ej. 429, 5xx); configurable con `Anthropic(max_retries=...)`.

### 8. Equivalencia rápida con OpenAI

| Concepto | OpenAI Responses | OpenAI Chat | Anthropic Messages |
|----------|------------------|-------------|---------------------|
| Rol / reglas globales | `instructions` | mensaje `system` | parámetro `system` |
| Hilo conversación | `input` (str o lista) | `messages` | `messages` (solo user/assistant) |
| Texto principal | `output_text` | `choices[0].message.content` | bloques `content` (p. ej. `.text`) |
| Tope salida | `max_output_tokens` | `max_tokens` (opcional) | **`max_tokens` (obligatorio)** |
| Fin / estado | `status`, … | `finish_reason` | `stop_reason` |
| Historial multi-turno | manual o `previous_response_id`+`store` | manual | **solo manual** |

### 9. Claves y `401`

- Clave en [Anthropic Console](https://console.anthropic.com/) (`sk-ant-...`).
- Secreto Colab: nombre exacto `ANTHROPIC_API_KEY`, sin espacios al pegar.


In [ ]:
# %pip install -q anthropic

import os
from getpass import getpass

from anthropic import (
    Anthropic,
    APIConnectionError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    RateLimitError,
)


def obtener_api_key_anthropic() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
    if key:
        return key
    return getpass("ANTHROPIC_API_KEY: ").strip()


# USD por 1M tokens (ejemplo Haiku 4.5 — actualiza en docs.anthropic.com / pricing)
PRICING_HAIKU = {"input": 1.00, "output": 5.00}


def texto_desde_content(content) -> str:
    partes = []
    for bloque in content:
        if getattr(bloque, "type", None) == "text" and hasattr(bloque, "text"):
            partes.append(bloque.text)
    return "".join(partes)


api_key = obtener_api_key_anthropic().strip()
if not api_key:
    raise ValueError("Falta ANTHROPIC_API_KEY")

client = Anthropic(api_key=api_key)
MODEL = "claude-haiku-4-5-20251001"

system_content = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

user_content = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

try:
    respuesta = client.messages.create(
        model=MODEL,
        system=system_content,
        messages=[{"role": "user", "content": user_content}],
        max_tokens=500,
        temperature=0.3,
    )
except AuthenticationError:
    print("401: revisa ANTHROPIC_API_KEY (clave sk-ant-... válida).")
    raise
except RateLimitError:
    print("429: rate limit o crédito; espera o revisa el panel de Anthropic.")
    raise
except BadRequestError as e:
    print("400:", getattr(e, "message", e))
    raise
except (APIConnectionError, InternalServerError) as e:
    print("Red o servidor Anthropic:", e)
    raise

texto = texto_desde_content(respuesta.content)

print("--- Contenido (bloques text en content) ---")
print(texto)

print("--- stop_reason ---")
print(respuesta.stop_reason)
if respuesta.stop_reason == "max_tokens":
    print("Aviso: respuesta truncada por max_tokens — sube el límite o divide la tarea.")
elif respuesta.stop_reason == "end_turn":
    print("Generación completada de forma natural.")

print("--- Metadatos ---")
print("id:", respuesta.id)
print("model:", respuesta.model)
print("type:", respuesta.type)
print("role:", respuesta.role)

if respuesta.usage:
    u = respuesta.usage
    print("usage:", u.model_dump())
    inp = u.input_tokens
    out = u.output_tokens
    total = inp + out
    print("total_tokens (calculado):", total)
    p = PRICING_HAIKU
    coste = (inp / 1_000_000) * p["input"] + (out / 1_000_000) * p["output"]
    print(f"coste aprox. (USD): ${coste:.6f}")

# Inspección opcional del JSON completo:
# print(respuesta.model_dump_json(indent=2)[:3000])

# --- Opcional: segundo turno con historial manual (stateless) ---
# r1 = client.messages.create(
#     model=MODEL,
#     system=system_content,
#     messages=[{"role": "user", "content": "¿Qué es una API REST?"}],
#     max_tokens=400,
#     temperature=0.3,
# )
# t1 = texto_desde_content(r1.content)
# r2 = client.messages.create(
#     model=MODEL,
#     system=system_content,
#     messages=[
#         {"role": "user", "content": "¿Qué es una API REST?"},
#         {"role": "assistant", "content": t1},
#         {"role": "user", "content": "¿En qué se diferencia de GraphQL?"},
#     ],
#     max_tokens=500,
#     temperature=0.3,
# )
# print(texto_desde_content(r2.content))
